# RescueNet ML — Google Colab training

Creates a **50,000-row** dataset and trains **two lightweight** models:

- Human food (kg / L / whole packets)
- Pet food for 10 dog breeds + 10 cat breeds (weight, breed, age, type)

**Why these versions:** Render and this notebook must share `numpy==1.26.4` and `scikit-learn==1.3.2`. Training on Colab's default NumPy 2.x caused pickle load failures.

**Model:** `HistGradientBoostingRegressor` inside `MultiOutputRegressor` — much smaller than one RandomForest pickle per food item, target **R² ≥ 0.90**.

1. Run the install cell. If Colab asks, **Restart session**, then run install again.
2. Run the remaining cells in order.
3. Download `rescunet_models.zip` and put `models/` + `food_schema.py` in `rescue-ml-api`.


In [1]:
# STEP 1 — pin versions to match Render (avoids numpy._core pickle errors)
!pip install -q numpy==1.26.4 pandas==2.1.4 scikit-learn==1.3.2 joblib==1.3.2
import numpy, sklearn, pandas, joblib
print('numpy', numpy.__version__)
print('sklearn', sklearn.__version__)
print('pandas', pandas.__version__)
print('joblib', joblib.__version__)
assert numpy.__version__.startswith('1.26'), 'Restart runtime after pip, then re-run this cell'
assert sklearn.__version__.startswith('1.3.2'), 'Restart runtime after pip, then re-run this cell'

You should consider upgrading via the 'c:\src\rescue\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


numpy 1.26.4
sklearn 1.3.2
pandas 2.1.4
joblib 1.3.2


In [2]:
from pathlib import Path
Path('food_schema.py').write_text('"""\nShared RescueNet food / pet schema used by dataset generation, training, and the API.\nKeep this file identical in Colab and on the server so predictions match training.\n"""\n\nINTEGER_UNITS = {"packets", "cans"}\n\nHUMAN_FOOD = {\n    "Rice": {"base": 0.4, "category": "staple", "unit": "kg", "needs_cooking": True},\n    "Dhal (lentils)": {"base": 0.15, "category": "protein", "unit": "kg", "needs_cooking": True},\n    "Cooking oil": {"base": 0.05, "category": "fat", "unit": "L", "needs_cooking": True},\n    "Salt": {"base": 0.02, "category": "spice", "unit": "kg", "needs_cooking": True},\n    "Sugar": {"base": 0.03, "category": "sweetener", "unit": "kg", "needs_cooking": True},\n    "Biscuits": {"base": 0.5, "category": "snack", "unit": "packets", "needs_cooking": False},\n    "Canned Tuna": {"base": 0.5, "category": "protein", "unit": "cans", "needs_cooking": False},\n    "Water": {"base": 3.0, "category": "water", "unit": "L", "needs_cooking": False},\n    "Milk powder": {"base": 0.05, "category": "dairy", "unit": "kg", "needs_cooking": False},\n    "Potatoes": {"base": 0.2, "category": "vegetable", "unit": "kg", "needs_cooking": True},\n    "Carrots": {"base": 0.1, "category": "vegetable", "unit": "kg", "needs_cooking": True},\n    "Cabbage": {"base": 0.15, "category": "vegetable", "unit": "kg", "needs_cooking": True},\n    "Pumpkin": {"base": 0.15, "category": "vegetable", "unit": "kg", "needs_cooking": True},\n    "Brinjal": {"base": 0.1, "category": "vegetable", "unit": "kg", "needs_cooking": True},\n    "Coconut": {"base": 0.2, "category": "fat", "unit": "kg", "needs_cooking": True},\n    "Bread": {"base": 0.3, "category": "staple", "unit": "kg", "needs_cooking": False},\n    "Noodles": {"base": 0.3, "category": "staple", "unit": "packets", "needs_cooking": False},\n    "Sandwiches": {"base": 0.2, "category": "snack", "unit": "packets", "needs_cooking": False},\n    "Boiled eggs": {"base": 0.2, "category": "protein", "unit": "packets", "needs_cooking": False},\n    "Soup": {"base": 0.3, "category": "snack", "unit": "packets", "needs_cooking": False},\n    "Rice and vegetable curry": {"base": 0.4, "category": "meal", "unit": "packets", "needs_cooking": False},\n    "Rice and chicken curry": {"base": 0.4, "category": "meal", "unit": "packets", "needs_cooking": False},\n    "Rice and canned fish": {"base": 0.4, "category": "meal", "unit": "packets", "needs_cooking": False},\n    "Vegetable fried rice": {"base": 0.5, "category": "meal", "unit": "packets", "needs_cooking": False},\n    "String hoppers with curry": {"base": 0.3, "category": "meal", "unit": "packets", "needs_cooking": False},\n    "Roti with curry": {"base": 0.3, "category": "meal", "unit": "packets", "needs_cooking": False},\n    "Paratha with curry": {"base": 0.3, "category": "meal", "unit": "packets", "needs_cooking": False},\n}\n\nPET_FOOD = {\n    "Dog Food": {"base": 0.3, "for": "dog", "unit": "kg"},\n    "Dog Treats": {"base": 0.05, "for": "dog", "unit": "packets"},\n    "Dog Biscuits": {"base": 0.1, "for": "dog", "unit": "packets"},\n    "Canned Dog Food": {"base": 0.4, "for": "dog", "unit": "cans"},\n    "Cat Food": {"base": 0.15, "for": "cat", "unit": "kg"},\n    "Cat Treats": {"base": 0.03, "for": "cat", "unit": "packets"},\n    "Canned Cat Food": {"base": 0.2, "for": "cat", "unit": "cans"},\n    "Cat Kibble": {"base": 0.12, "for": "cat", "unit": "kg"},\n    "Pet Milk": {"base": 0.1, "for": "both", "unit": "L"},\n    "Pet Supplements": {"base": 0.02, "for": "both", "unit": "packets"},\n}\n\nDOG_BREEDS = [\n    "Labrador Retriever",\n    "German Shepherd",\n    "Golden Retriever",\n    "Bulldog",\n    "Poodle",\n    "Beagle",\n    "Rottweiler",\n    "Dachshund",\n    "Siberian Husky",\n    "Great Dane",\n]\n\nCAT_BREEDS = [\n    "Persian",\n    "Siamese",\n    "Maine Coon",\n    "Ragdoll",\n    "Bengal",\n    "Sphynx",\n    "British Shorthair",\n    "Abyssinian",\n    "Scottish Fold",\n    "Oriental Shorthair",\n]\n\nDOG_BREED_PROFILE = {\n    "Labrador Retriever": {"w_min": 25.0, "w_max": 36.0, "factor": 1.10},\n    "German Shepherd": {"w_min": 22.0, "w_max": 40.0, "factor": 1.15},\n    "Golden Retriever": {"w_min": 25.0, "w_max": 34.0, "factor": 1.10},\n    "Bulldog": {"w_min": 18.0, "w_max": 25.0, "factor": 0.90},\n    "Poodle": {"w_min": 6.0, "w_max": 20.0, "factor": 0.70},\n    "Beagle": {"w_min": 9.0, "w_max": 11.0, "factor": 0.60},\n    "Rottweiler": {"w_min": 35.0, "w_max": 50.0, "factor": 1.30},\n    "Dachshund": {"w_min": 7.0, "w_max": 12.0, "factor": 0.45},\n    "Siberian Husky": {"w_min": 16.0, "w_max": 27.0, "factor": 1.00},\n    "Great Dane": {"w_min": 45.0, "w_max": 80.0, "factor": 1.60},\n}\n\nCAT_BREED_PROFILE = {\n    "Persian": {"w_min": 3.5, "w_max": 5.5, "factor": 1.00},\n    "Siamese": {"w_min": 2.5, "w_max": 5.0, "factor": 0.85},\n    "Maine Coon": {"w_min": 5.0, "w_max": 8.5, "factor": 1.40},\n    "Ragdoll": {"w_min": 4.5, "w_max": 7.5, "factor": 1.20},\n    "Bengal": {"w_min": 3.5, "w_max": 6.5, "factor": 1.05},\n    "Sphynx": {"w_min": 3.0, "w_max": 5.0, "factor": 0.80},\n    "British Shorthair": {"w_min": 4.0, "w_max": 7.0, "factor": 1.10},\n    "Abyssinian": {"w_min": 3.0, "w_max": 4.5, "factor": 0.85},\n    "Scottish Fold": {"w_min": 3.5, "w_max": 6.0, "factor": 0.95},\n    "Oriental Shorthair": {"w_min": 2.5, "w_max": 4.5, "factor": 0.80},\n}\n\nEMERGENCY_TYPES = [\n    "Flood",\n    "Earthquake",\n    "Tsunami",\n    "Cyclone",\n    "Landslide",\n    "Fire",\n    "Drought",\n    "Epidemic",\n]\n\nEMERGENCY_MULTIPLIERS = {\n    "Flood": 1.20,\n    "Earthquake": 1.30,\n    "Tsunami": 1.40,\n    "Cyclone": 1.25,\n    "Landslide": 1.30,\n    "Fire": 1.10,\n    "Drought": 1.15,\n    "Epidemic": 1.30,\n}\n\nFEATURE_COLUMNS = [\n    "emergency_type_encoded",\n    "total_people",\n    "cooking_available",\n    "children",\n    "elderly",\n    "pregnant",\n    "vegetarian_count",\n    "diabetes_count",\n    "bp_count",\n    "heart_count",\n    "lactating_count",\n    "days_required",\n    "pet_count",\n    "dog_count",\n    "cat_count",\n    "total_dog_weight_kg",\n    "total_cat_weight_kg",\n    "avg_dog_age",\n    "avg_cat_age",\n    "dog_size_load",\n    "cat_size_load",\n    "puppy_count",\n    "kitten_count",\n    "senior_dog_count",\n    "senior_cat_count",\n]\n\nHUMAN_TARGETS = list(HUMAN_FOOD.keys())\nPET_TARGETS = list(PET_FOOD.keys())\n\n\ndef normalize_pet_type(value):\n    text = str(value or "dog").strip().lower()\n    if text.startswith("cat"):\n        return "cat"\n    return "dog"\n\n\ndef breed_factor(pet_type, breed):\n    breed = str(breed or "").strip()\n    if pet_type == "dog":\n        return DOG_BREED_PROFILE.get(breed, {}).get("factor", 1.0)\n    return CAT_BREED_PROFILE.get(breed, {}).get("factor", 1.0)\n\n\ndef age_factor(pet_type, age):\n    age = float(age or 0)\n    if pet_type == "dog":\n        if age < 1:\n            return 1.30\n        if age >= 8:\n            return 0.85\n        return 1.0\n    if age < 1:\n        return 1.35\n    if age >= 10:\n        return 0.85\n    return 1.0\n\n\ndef pet_size_load(pet):\n    pet_type = normalize_pet_type(pet.get("type"))\n    weight = max(0.0, float(pet.get("weight") or 0))\n    age = float(pet.get("age") or 0)\n    return weight * breed_factor(pet_type, pet.get("breed")) * age_factor(pet_type, age)\n\n\ndef round_amount(amount, unit):\n    amount = max(0.0, float(amount))\n    if unit in INTEGER_UNITS:\n        return int(round(amount))\n    return round(amount, 2)\n\n\ndef cooking_multiplier(needs_cooking, cooking_available, category):\n    if category == "water":\n        return 1.0\n    if cooking_available:\n        if category == "meal":\n            return 0.22\n        if needs_cooking:\n            return 1.0\n        return 0.35\n    if needs_cooking:\n        return 0.12\n    if category == "meal":\n        return 1.45\n    return 1.25\n', encoding='utf-8')
print('Wrote food_schema.py')

Wrote food_schema.py


In [3]:
from pathlib import Path
Path('generate_dataset.py').write_text('"""\nGenerate 50,000 emergency food rows for RescueNet.\nRun locally or in Google Colab after installing pinned numpy/pandas.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nfrom datetime import datetime, timedelta\n\nimport numpy as np\nimport pandas as pd\n\nfrom food_schema import (\n    CAT_BREED_PROFILE,\n    CAT_BREEDS,\n    DOG_BREED_PROFILE,\n    DOG_BREEDS,\n    EMERGENCY_MULTIPLIERS,\n    EMERGENCY_TYPES,\n    FEATURE_COLUMNS,\n    HUMAN_FOOD,\n    HUMAN_TARGETS,\n    PET_FOOD,\n    PET_TARGETS,\n    age_factor,\n    breed_factor,\n    cooking_multiplier,\n    normalize_pet_type,\n    pet_size_load,\n    round_amount,\n)\n\nN_ROWS = 50000\nRANDOM_SEED = 42\nCSV_NAME = "emergency_food_50000_v3.csv"\n\n\ndef generate_person(idx: int) -> dict:\n    age = int(np.random.randint(1, 86))\n    gender = random.choice(["Male", "Female", "Other"])\n    dietary = random.choice(["None", "Vegetarian", "Vegan", "Gluten-Free", "Halal", "Kosher"])\n    health = random.choice(\n        ["None", "None", "None", "Diabetes", "High Blood Pressure", "Heart Condition", "Lactating"]\n    )\n    is_pregnant = bool(gender == "Female" and 18 <= age <= 45 and random.random() < 0.12)\n    if is_pregnant:\n        health = "Pregnant"\n    return {\n        "person_id": f"P{idx:04d}",\n        "age": age,\n        "gender": gender,\n        "isPregnant": is_pregnant,\n        "dietaryRestriction": dietary,\n        "healthCondition": health,\n    }\n\n\ndef generate_pet(idx: int, pet_type: str) -> dict:\n    if pet_type == "dog":\n        breed = random.choice(DOG_BREEDS)\n        profile = DOG_BREED_PROFILE[breed]\n        age = int(np.random.randint(0, 13))\n        weight = round(float(np.random.uniform(profile["w_min"], profile["w_max"])), 1)\n        dietary = random.choice(["Standard", "Senior", "Puppy/Kitten", "Weight Management", "Allergy-Prone"])\n    else:\n        breed = random.choice(CAT_BREEDS)\n        profile = CAT_BREED_PROFILE[breed]\n        age = int(np.random.randint(0, 16))\n        weight = round(float(np.random.uniform(profile["w_min"], profile["w_max"])), 1)\n        dietary = random.choice(["Standard", "Senior", "Puppy/Kitten", "Weight Management", "Allergy-Prone"])\n    return {\n        "pet_id": f"PET{idx:04d}",\n        "type": pet_type.capitalize(),\n        "breed": breed,\n        "age": age,\n        "weight": weight,\n        "dietaryNeed": dietary,\n    }\n\n\ndef summarize_people(people: list[dict]) -> dict:\n    children = sum(1 for p in people if p["age"] < 12)\n    elderly = sum(1 for p in people if p["age"] > 60)\n    pregnant = sum(1 for p in people if p["isPregnant"] or p["healthCondition"] == "Pregnant")\n    vegetarian = sum(1 for p in people if p["dietaryRestriction"] in ("Vegetarian", "Vegan"))\n    diabetes = sum(1 for p in people if p["healthCondition"] == "Diabetes")\n    bp = sum(1 for p in people if p["healthCondition"] == "High Blood Pressure")\n    heart = sum(1 for p in people if p["healthCondition"] == "Heart Condition")\n    lactating = sum(1 for p in people if p["healthCondition"] == "Lactating")\n    return {\n        "children": children,\n        "elderly": elderly,\n        "pregnant": pregnant,\n        "vegetarian_count": vegetarian,\n        "diabetes_count": diabetes,\n        "bp_count": bp,\n        "heart_count": heart,\n        "lactating_count": lactating,\n    }\n\n\ndef summarize_pets(pets: list[dict]) -> dict:\n    dogs = [p for p in pets if normalize_pet_type(p.get("type")) == "dog"]\n    cats = [p for p in pets if normalize_pet_type(p.get("type")) == "cat"]\n    dog_weights = [float(p["weight"]) for p in dogs]\n    cat_weights = [float(p["weight"]) for p in cats]\n    dog_ages = [float(p["age"]) for p in dogs]\n    cat_ages = [float(p["age"]) for p in cats]\n    return {\n        "pet_count": len(pets),\n        "dog_count": len(dogs),\n        "cat_count": len(cats),\n        "total_dog_weight_kg": round(sum(dog_weights), 2),\n        "total_cat_weight_kg": round(sum(cat_weights), 2),\n        "avg_dog_age": round(sum(dog_ages) / len(dog_ages), 2) if dog_ages else 0.0,\n        "avg_cat_age": round(sum(cat_ages) / len(cat_ages), 2) if cat_ages else 0.0,\n        "dog_size_load": round(sum(pet_size_load(p) for p in dogs), 3),\n        "cat_size_load": round(sum(pet_size_load(p) for p in cats), 3),\n        "puppy_count": sum(1 for p in dogs if float(p["age"]) < 1),\n        "kitten_count": sum(1 for p in cats if float(p["age"]) < 1),\n        "senior_dog_count": sum(1 for p in dogs if float(p["age"]) >= 8),\n        "senior_cat_count": sum(1 for p in cats if float(p["age"]) >= 10),\n        "dog_breeds": "|".join(p["breed"] for p in dogs),\n        "cat_breeds": "|".join(p["breed"] for p in cats),\n    }\n\n\ndef human_food_amounts(total_people: int, days: int, cooking_available: bool, multiplier: float, stats: dict) -> dict:\n    values = {}\n    n = max(1, total_people)\n    for food, spec in HUMAN_FOOD.items():\n        amount = spec["base"] * total_people * days * multiplier\n        amount *= cooking_multiplier(spec["needs_cooking"], cooking_available, spec["category"])\n\n        if food == "Sugar":\n            amount *= 1 - 0.75 * (stats["diabetes_count"] / n)\n        if food == "Salt":\n            amount *= 1 - 0.80 * ((stats["bp_count"] + stats["heart_count"]) / n)\n        if food == "Cooking oil":\n            amount *= 1 - 0.25 * (stats["heart_count"] / n)\n        if food == "Milk powder":\n            amount *= 1 + 0.55 * ((stats["pregnant"] + stats["lactating_count"] + stats["children"]) / n)\n        if food == "Dhal (lentils)":\n            amount *= 1 + 0.35 * (stats["vegetarian_count"] / n)\n        if food == "Canned Tuna" or food == "Rice and chicken curry" or food == "Rice and canned fish":\n            amount *= 1 - 0.85 * (stats["vegetarian_count"] / n)\n        if food in ("Biscuits", "Milk powder", "Soup"):\n            amount *= 1 + 0.25 * (stats["children"] / n)\n        if food in ("Soup", "Rice"):\n            amount *= 1 + 0.18 * (stats["elderly"] / n)\n\n        amount *= float(np.random.uniform(0.97, 1.03))\n        values[food] = round_amount(amount, spec["unit"])\n    return values\n\n\ndef pet_food_amounts(pets: list[dict], days: int, multiplier: float) -> dict:\n    values = {}\n    for food, spec in PET_FOOD.items():\n        total = 0.0\n        for pet in pets:\n            pet_type = normalize_pet_type(pet.get("type"))\n            if spec["for"] not in (pet_type, "both"):\n                continue\n            ref = 20.0 if pet_type == "dog" else 5.0\n            weight = max(0.0, float(pet.get("weight") or 0))\n            load = (weight / ref) * breed_factor(pet_type, pet.get("breed")) * age_factor(pet_type, pet.get("age"))\n            total += spec["base"] * load * days * (0.92 + 0.08 * multiplier)\n        total *= float(np.random.uniform(0.97, 1.03)) if total > 0 else 0.0\n        values[food] = round_amount(total, spec["unit"])\n    return values\n\n\ndef generate_dataset(n_rows: int = N_ROWS, seed: int = RANDOM_SEED) -> pd.DataFrame:\n    np.random.seed(seed)\n    random.seed(seed)\n\n    encoder_map = {name: i for i, name in enumerate(EMERGENCY_TYPES)}\n    rows = []\n    pet_id = 0\n    locations = [\n        "Colombo", "Kandy", "Galle", "Jaffna", "Matara", "Negombo", "Anuradhapura",\n        "Polonnaruwa", "Badulla", "Ratnapura", "Kurunegala", "Trincomalee", "Batticaloa",\n    ]\n\n    for i in range(n_rows):\n        if (i + 1) % 10000 == 0:\n            print(f"  Progress: {i + 1}/{n_rows}")\n\n        emergency = random.choice(EMERGENCY_TYPES)\n        multiplier = EMERGENCY_MULTIPLIERS[emergency]\n        total_people = int(np.random.randint(1, 16))\n        people = [generate_person(p + 1) for p in range(total_people)]\n        stats = summarize_people(people)\n        cooking_available = bool(random.random() < 0.55)\n        days = int(np.random.randint(2, 8))\n\n        pets = []\n        if random.random() < 0.48:\n            for _ in range(int(np.random.randint(1, 4))):\n                pet_id += 1\n                pets.append(generate_pet(pet_id, random.choice(["dog", "cat"])))\n        pet_stats = summarize_pets(pets)\n\n        human_vals = human_food_amounts(total_people, days, cooking_available, multiplier, stats)\n        pet_vals = pet_food_amounts(pets, days, multiplier)\n\n        timestamp = datetime(2024, 1, 1) + timedelta(\n            days=int(np.random.randint(0, 366)),\n            hours=int(np.random.randint(0, 24)),\n        )\n\n        row = {\n            "emergency_type": emergency,\n            "emergency_type_encoded": encoder_map[emergency],\n            "total_people": total_people,\n            "cooking_available": int(cooking_available),\n            "days_required": days,\n            "location": random.choice(locations),\n            "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),\n            **stats,\n            **{k: v for k, v in pet_stats.items()},\n            **{f"human_{name}": human_vals[name] for name in HUMAN_TARGETS},\n            **{f"pet_{name}": pet_vals[name] for name in PET_TARGETS},\n        }\n        rows.append(row)\n\n    df = pd.DataFrame(rows)\n    df = df.fillna(0)\n    return df\n\n\ndef main():\n    print("=" * 72)\n    print("Generating RescueNet 50,000-row dataset")\n    print("=" * 72)\n    df = generate_dataset()\n    out_path = os.path.join(os.path.dirname(__file__), CSV_NAME)\n    df.to_csv(out_path, index=False)\n\n    meta = {\n        "rows": int(len(df)),\n        "human_foods": HUMAN_TARGETS,\n        "pet_foods": PET_TARGETS,\n        "features": FEATURE_COLUMNS,\n        "dog_breeds": DOG_BREEDS,\n        "cat_breeds": CAT_BREEDS,\n        "emergency_types": EMERGENCY_TYPES,\n        "csv": CSV_NAME,\n    }\n    with open(os.path.join(os.path.dirname(__file__), "dataset_meta.json"), "w", encoding="utf-8") as f:\n        json.dump(meta, f, indent=2)\n\n    print(f"Saved {len(df)} rows -> {out_path}")\n    print(f"Pet households: {(df[\'pet_count\'] > 0).sum()}")\n    print(f"Human food columns: {len(HUMAN_TARGETS)}")\n    print(f"Pet food columns: {len(PET_TARGETS)}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote generate_dataset.py')

Wrote generate_dataset.py


In [4]:
from pathlib import Path
Path('train_models.py').write_text('"""\nTrain lightweight HistGradientBoosting models for RescueNet.\nPin numpy==1.26.4 and scikit-learn==1.3.2 in Colab AND on Render so pickles load.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\nimport warnings\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.multioutput import MultiOutputRegressor\nfrom sklearn.preprocessing import LabelEncoder\n\nfrom food_schema import (\n    CAT_BREEDS,\n    DOG_BREEDS,\n    EMERGENCY_TYPES,\n    FEATURE_COLUMNS,\n    HUMAN_FOOD,\n    HUMAN_TARGETS,\n    PET_FOOD,\n    PET_TARGETS,\n)\n\nwarnings.filterwarnings("ignore")\n\nCSV_NAME = "emergency_food_50000_v3.csv"\nMODEL_DIR = "models"\n\n\ndef evaluate(y_true: pd.DataFrame, y_pred: np.ndarray, names: list[str]) -> dict:\n    results = {}\n    r2_list = []\n    mape_list = []\n    for i, name in enumerate(names):\n        yt = y_true.iloc[:, i].to_numpy(dtype=float)\n        yp = y_pred[:, i]\n        r2 = float(r2_score(yt, yp))\n        mae = float(mean_absolute_error(yt, yp))\n        rmse = float(np.sqrt(mean_squared_error(yt, yp)))\n        mask = yt != 0\n        mape = float(np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100) if np.any(mask) else 0.0\n        results[name] = {"R2": r2, "MAE": mae, "RMSE": rmse, "MAPE": mape}\n        r2_list.append(r2)\n        mape_list.append(mape)\n        print(f"  {name:32s}  R2={r2:.4f}  MAPE={mape:5.1f}%  MAE={mae:.3f}")\n    return {\n        "per_target": results,\n        "avg_r2": float(np.mean(r2_list)),\n        "avg_mape": float(np.mean(mape_list)),\n        "min_r2": float(np.min(r2_list)),\n    }\n\n\ndef make_regressor() -> HistGradientBoostingRegressor:\n    return HistGradientBoostingRegressor(\n        max_iter=140,\n        learning_rate=0.08,\n        max_depth=6,\n        max_leaf_nodes=31,\n        min_samples_leaf=20,\n        l2_regularization=0.05,\n        early_stopping=False,\n        random_state=42,\n    )\n\n\ndef main():\n    started = time.time()\n    base = os.path.dirname(__file__)\n    csv_path = os.path.join(base, CSV_NAME)\n    model_dir = os.path.join(base, MODEL_DIR)\n    os.makedirs(model_dir, exist_ok=True)\n\n    print("=" * 72)\n    print("RescueNet model training")\n    print(f"  numpy={np.__version__}")\n    import sklearn\n\n    print(f"  sklearn={sklearn.__version__}")\n    print("  model=HistGradientBoostingRegressor + MultiOutputRegressor")\n    print("=" * 72)\n\n    if not os.path.exists(csv_path):\n        raise FileNotFoundError(f"Dataset not found: {csv_path}. Run generate_dataset.py first.")\n\n    df = pd.read_csv(csv_path)\n    df = df.fillna(0).drop_duplicates()\n    print(f"Loaded {len(df)} rows")\n\n    le = LabelEncoder()\n    le.fit(EMERGENCY_TYPES)\n    df["emergency_type_encoded"] = le.transform(df["emergency_type"])\n\n    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]\n    if missing:\n        raise ValueError(f"Missing feature columns: {missing}")\n\n    X = df[FEATURE_COLUMNS]\n    y_human = df[[f"human_{name}" for name in HUMAN_TARGETS]]\n    y_pet = df[[f"pet_{name}" for name in PET_TARGETS]]\n\n    X_train, X_test, yh_train, yh_test, yp_train, yp_test = train_test_split(\n        X, y_human, y_pet, test_size=0.2, random_state=42\n    )\n    print(f"Train={len(X_train)}  Test={len(X_test)}")\n\n    print("\\nTraining human food model...")\n    human_model = MultiOutputRegressor(make_regressor(), n_jobs=-1)\n    human_model.fit(X_train, yh_train)\n    human_pred = human_model.predict(X_test)\n    human_scores = evaluate(yh_test, human_pred, HUMAN_TARGETS)\n\n    print("\\nTraining pet food model...")\n    pet_model = MultiOutputRegressor(make_regressor(), n_jobs=-1)\n    pet_model.fit(X_train, yp_train)\n    pet_pred = pet_model.predict(X_test)\n    pet_scores = evaluate(yp_test, pet_pred, PET_TARGETS)\n\n    joblib.dump(human_model, os.path.join(model_dir, "human_food_model.pkl"))\n    joblib.dump(pet_model, os.path.join(model_dir, "pet_food_model.pkl"))\n    joblib.dump(le, os.path.join(model_dir, "label_encoder.pkl"))\n\n    with open(os.path.join(model_dir, "feature_names.txt"), "w", encoding="utf-8") as f:\n        f.write(",".join(FEATURE_COLUMNS))\n\n    food_info = {\n        "human": {name: {"unit": spec["unit"], "needs_cooking": spec["needs_cooking"]} for name, spec in HUMAN_FOOD.items()},\n        "pets": {name: {"unit": spec["unit"], "for": spec["for"]} for name, spec in PET_FOOD.items()},\n        "dog_breeds": DOG_BREEDS,\n        "cat_breeds": CAT_BREEDS,\n        "emergency_types": EMERGENCY_TYPES,\n    }\n    with open(os.path.join(model_dir, "food_info.json"), "w", encoding="utf-8") as f:\n        json.dump(food_info, f, indent=2)\n\n    metadata = {\n        "model_type": "HistGradientBoostingRegressor",\n        "wrapper": "MultiOutputRegressor",\n        "sklearn_version": sklearn.__version__,\n        "numpy_version": np.__version__,\n        "features": FEATURE_COLUMNS,\n        "human_food_targets": HUMAN_TARGETS,\n        "pet_food_targets": PET_TARGETS,\n        "human_avg_r2": human_scores["avg_r2"],\n        "human_avg_mape": human_scores["avg_mape"],\n        "human_min_r2": human_scores["min_r2"],\n        "pet_avg_r2": pet_scores["avg_r2"],\n        "pet_avg_mape": pet_scores["avg_mape"],\n        "pet_min_r2": pet_scores["min_r2"],\n        "data_rows": int(len(df)),\n        "training_rows": int(len(X_train)),\n        "testing_rows": int(len(X_test)),\n        "human_results": human_scores["per_target"],\n        "pet_results": pet_scores["per_target"],\n        "notes": "Train with numpy==1.26.4 and scikit-learn==1.3.2. Copy models/ onto Render.",\n    }\n    with open(os.path.join(model_dir, "metadata.json"), "w", encoding="utf-8") as f:\n        json.dump(metadata, f, indent=2)\n\n    elapsed = time.time() - started\n    print("\\n" + "=" * 72)\n    print(f"Human avg R2: {human_scores[\'avg_r2\']:.4f}  min R2: {human_scores[\'min_r2\']:.4f}")\n    print(f"Pet    avg R2: {pet_scores[\'avg_r2\']:.4f}  min R2: {pet_scores[\'min_r2\']:.4f}")\n    if human_scores["avg_r2"] >= 0.90 and pet_scores["avg_r2"] >= 0.90:\n        print("Target accuracy 90%+ met.")\n    else:\n        print("Accuracy below 90%. Reduce dataset noise or increase max_iter.")\n    print(f"Saved models in {model_dir}  ({elapsed/60:.1f} min)")\n    print("=" * 72)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote train_models.py')

Wrote train_models.py


In [5]:
# STEP 5 — generate 50,000 rows (kg / L / whole packets)
from generate_dataset import main as generate_main
generate_main()
import pandas as pd
df = pd.read_csv('emergency_food_50000_v3.csv')
print(df.shape)
print('Human foods', [c for c in df.columns if c.startswith('human_')])
print('Pet foods', [c for c in df.columns if c.startswith('pet_')])
print(df[['total_people','cooking_available','days_required','pet_count','dog_size_load','human_Rice','human_Water','pet_Dog Food']].head())

Generating RescueNet 50,000-row dataset
  Progress: 10000/50000
  Progress: 20000/50000
  Progress: 30000/50000
  Progress: 40000/50000
  Progress: 50000/50000
Saved 50000 rows -> c:\src\rescue\rescue-ml-api\emergency_food_50000_v3.csv
Pet households: 23875
Human food columns: 27
Pet food columns: 10
(50000, 67)
Human foods ['human_Rice', 'human_Dhal (lentils)', 'human_Cooking oil', 'human_Salt', 'human_Sugar', 'human_Biscuits', 'human_Canned Tuna', 'human_Water', 'human_Milk powder', 'human_Potatoes', 'human_Carrots', 'human_Cabbage', 'human_Pumpkin', 'human_Brinjal', 'human_Coconut', 'human_Bread', 'human_Noodles', 'human_Sandwiches', 'human_Boiled eggs', 'human_Soup', 'human_Rice and vegetable curry', 'human_Rice and chicken curry', 'human_Rice and canned fish', 'human_Vegetable fried rice', 'human_String hoppers with curry', 'human_Roti with curry', 'human_Paratha with curry']
Pet foods ['pet_count', 'pet_Dog Food', 'pet_Dog Treats', 'pet_Dog Biscuits', 'pet_Canned Dog Food', 'pet_

In [6]:
# STEP 6 — train two multi-output HistGradientBoosting models
from train_models import main as train_main
train_main()
import json
from pathlib import Path
meta = json.loads(Path('models/metadata.json').read_text())
print('Human avg R2', round(meta['human_avg_r2'], 4), 'min', round(meta['human_min_r2'], 4))
print('Pet    avg R2', round(meta['pet_avg_r2'], 4), 'min', round(meta['pet_min_r2'], 4))

RescueNet model training
  numpy=1.26.4
  sklearn=1.3.2
  model=HistGradientBoostingRegressor + MultiOutputRegressor
Loaded 50000 rows
Train=40000  Test=10000

Training human food model...
  Rice                              R2=0.9993  MAPE=  3.2%  MAE=0.206
  Dhal (lentils)                    R2=0.9993  MAPE=  3.2%  MAE=0.085
  Cooking oil                       R2=0.9993  MAPE=  3.3%  MAE=0.023
  Salt                              R2=0.9986  MAPE=  6.9%  MAE=0.011
  Sugar                             R2=0.9992  MAPE=  4.1%  MAE=0.015
  Biscuits                          R2=0.9989  MAPE=  4.7%  MAE=0.414
  Canned Tuna                       R2=0.9983  MAPE=  6.3%  MAE=0.379
  Water                             R2=0.9990  MAPE=  1.7%  MAE=2.117
  Milk powder                       R2=0.9987  MAPE=  3.8%  MAE=0.048
  Potatoes                          R2=0.9994  MAPE=  2.5%  MAE=0.090
  Carrots                           R2=0.9994  MAPE=  2.7%  MAE=0.046
  Cabbage                           R2=0.

In [7]:
# STEP 7 — zip models for download / Render deploy
import shutil
from google.colab import files
shutil.make_archive('rescunet_models', 'zip', 'models')
print('Copy models/ and food_schema.py into rescue-ml-api, then redeploy Render.')
files.download('rescunet_models.zip')
files.download('food_schema.py')
files.download('emergency_food_50000_v3.csv')

ModuleNotFoundError: No module named 'google'